#LAB ASSIGNMENT / 03
Text Generation Using GPT 2

##Step 1 Load the Model

In [4]:
# Colab cell before imports: %pip install -q "transformers>=4.51,<5"
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

##Step 2 Compare Prompts and Decoding

In [6]:
def continue_text(prompt, *, sample=False, temperature=0.8, seed=42):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    options = {"do_sample": sample, "max_new_tokens": 45,
               "pad_token_id": tokenizer.eos_token_id}
    if sample:
        options.update(temperature=temperature, top_p=0.9)
    with torch.no_grad():
        tokens = model.generate(**inputs, **options)
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):],
                            skip_special_tokens=True).strip()

prompts = [
    "A story about a robot begins like this:",
    "In a school library, a tiny robot found a lost book. It",
]
for prompt in prompts:
    print("\nPROMPT:", prompt)
    print("GREEDY:", continue_text(prompt))
    for temp in (0.6, 1.0):
        for seed in (42, 43):
            print(temp, seed, continue_text(prompt, sample=True,
                                           temperature=temp, seed=seed))


PROMPT: A story about a robot begins like this:
GREEDY: A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed
0.6 42 The robot was a humanoid robot with a heart, eyes, and hands that looked like a human, but its skin was red. The robot had a small, muscular body and was very muscular. It was also very slow.
0.6 43 The robot, a robot named "The Robot," is trying to figure out what happened to its owner, who is a woman. The robot tries to figure out how to get back to her, and then it has to figure
1.0 42 The Robot Man was brought to Earth by the Galactic government, to escape the dark and deadly dangers of space. The robot is the most powerful of all the robots, and even better. It has a long memory for what was
1.0 43 The person in charge of the robots, his robot friend, or even their dog, is trying to be good. But as soon as their d

##Step 3 Fine Tune on Study Tip Examples

In [8]:
from pathlib import Path
out = Path(".")

# For the full activity, add varied examples from your approved dataset.
examples = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
]
tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)
# Freeze the base model and train only the last Transformer block.
tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=2e-5)

for epoch in range(3):
    tuned.train()
    for topic, answer in examples:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
    print("Epoch:", epoch + 1, "last example loss:", round(loss.item(), 3))

tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch: 1 last example loss: 4.991
Epoch: 2 last example loss: 4.534
Epoch: 3 last example loss: 4.832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('study_tip_gpt2/tokenizer_config.json', 'study_tip_gpt2/tokenizer.json')

##Step 4 Compare on Held Out Topics

In [10]:
held_out = ["recursion", "matrices", "binary search"]
base_model = model
for topic in held_out:
    prompt = f"Topic: {topic}\nStudy tip:"
    model = base_model
    print(topic, "base:", continue_text(prompt))
    model = tuned.eval()
    print(topic, "tuned:", continue_text(prompt))
    model = base_model

recursion base: Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int i) { return i; } public: Recursion(int i)
recursion tuned: Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int x, int y) { return x + y; } public: Rec
matrices base: Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
matrices tuned: Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
binary search base: Use the search box to find the search term you want to search for.

Study tip: Use the search box to find the search term you want to search for.

Study tip: Use the search box to
binary search tuned: Use the search box to find the search